In [10]:
# =============================================================================
# MassBank MS2 Spectrum Extraction Toolkit — Interactive (Single-Notebook Cell)
# =============================================================================
# Author: Sergio Pérez Tabero (UAM)
#
# Overview
# --------
# This Jupyter/ipywidgets tool helps you:
#   1) Search MassBank MS2 records by compound name (via MassBank API).
#   2) Load canonical MassBank record files (.txt) from the MassBank data archive
#      (either a local ZIP or the GitHub ZIP downloaded in-memory).
#   3) Parse PK$PEAK to extract m/z and relative intensities.
#   4) Parse PK$ANNOTATION to map fragment formulas to peaks (ppm matching).
#   5) Export one .dat per accession + a per-molecule README log with metadata.
#
# Notes
# -----
# - The UI requires Jupyter Notebook/Lab (ipywidgets). The core functions
#   (download, parse, export) can be reused in a command-line script.
#
# =============================================================================


# -------------------------
# User configuration
# -------------------------
DEFAULT_MOLECULE = "Triadimenol"
PPM_TOL_DEFAULT  = 5.0
OUT_DIR          = "massbank_dat"

# Leave empty to download the MassBank-data ZIP from GitHub (in-memory).
# Example local path: "/path/to/MassBank-data-main.zip"
LOCAL_ZIP_PATH   = ""

GITHUB_ZIP       = "https://github.com/MassBank/MassBank-data/archive/refs/heads/main.zip"


# -------------------------
# Imports
# -------------------------
%matplotlib widget

import os

#import os
import io
from IPython.display import Image, display
import re
import zipfile
import gzip
import json as _json
import sys
import datetime as _dt
import numpy as np

from typing import Optional, Dict, List, Tuple, Any

from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as W

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import plotly.graph_objects as go
import mplcursors

# -------------------------
# Python version guard
# -------------------------
if sys.version_info < (3, 8):
    raise RuntimeError(
        "This notebook requires Python >= 3.8. "
        f"Your current version is {sys.version.split()[0]}."
    )


# =============================================================================
# Networking helpers
# =============================================================================

UA: Dict[str, str] = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko)"
}

def http_get(url: str, timeout: int = 30, headers: Optional[Dict[str, str]] = None) -> bytes:
    """
    Download raw bytes from a URL (HTTP GET).

    - Adds a browser-like User-Agent by default.
    - Transparently decompresses gzip responses if the server sends them.
    - Returns the response body as bytes.
    """
    h = dict(UA)
    if headers:
        h.update(headers)

    req = Request(url, headers=h)

    with urlopen(req, timeout=timeout) as r:
        data = r.read()
        enc = (r.headers.get("Content-Encoding") or "").lower()
        if enc == "gzip":
            data = gzip.GzipFile(fileobj=io.BytesIO(data)).read()
        return data

def http_get_json(url: str, **kw):
    """
    Download a URL and parse it as JSON.

    Returns:
        - Python object (list/dict) if JSON decoding works
        - None otherwise
    """
    data = http_get(url, **kw)
    try:
        return _json.loads(data.decode("utf-8", errors="replace"))
    except Exception:
        return None


# =============================================================================
# MassBank API search (optional convenience)
# =============================================================================

API_EU   = "https://massbank.eu/MassBank-api/records"
API_MSBI = "https://msbi.ipb-halle.de/MassBank-api/records"

def api_search_ms2_by_name(name: str) -> List[Dict[str, Any]]:
    """
    Search MassBank APIs for MS2 records matching a compound name.

    Args:
        name: Compound name to search.

    Returns:
        A list of dicts: [{"accession": "...", "title": "..."}, ...]
        Duplicates (same accession) are removed.
    """
    params = urlencode({"compound_name": name, "ms_type": "MS2"})
    accs: List[Dict[str, Any]] = []

    for base in (API_EU, API_MSBI):
        try:
            data = http_get_json(f"{base}?{params}", timeout=30) or []
        except Exception:
            data = []

        if isinstance(data, list):
            for rec in data:
                acc = rec.get("accession") or rec.get("Accession") or rec.get("id")
                title = rec.get("title") or rec.get("compound_name") or ""
                if acc:
                    accs.append({"accession": acc, "title": title})

    seen = set()
    out: List[Dict[str, Any]] = []
    for d in accs:
        if d["accession"] in seen:
            continue
        seen.add(d["accession"])
        out.append(d)

    return out


# =============================================================================
# MassBank-data ZIP helpers
# =============================================================================

def open_massbank_zip(local_zip: Optional[str] = None) -> zipfile.ZipFile:
    """
    Open the MassBank-data ZIP archive.

    - If local_zip is provided and exists, opens it from disk.
    - Otherwise downloads the official GitHub ZIP into memory and opens it.

    Returns:
        An open zipfile.ZipFile handle.
    """
    if local_zip and os.path.exists(local_zip):
        return zipfile.ZipFile(local_zip, "r")

    mem = http_get(GITHUB_ZIP, timeout=60)
    try:
        return zipfile.ZipFile(io.BytesIO(mem), "r")
    except Exception:
        # Last attempt: re-download once
        mem2 = http_get(GITHUB_ZIP, timeout=60)
        return zipfile.ZipFile(io.BytesIO(mem2), "r")

def read_accession_txt(zf: zipfile.ZipFile, accession: str) -> str:
    """
    Read the MassBank record text file for a given accession from the ZIP.

    Args:
        zf: Open zipfile handle.
        accession: MassBank accession (e.g., "PR010001").

    Returns:
        The record content as a UTF-8 string (errors replaced).
    """
    target = f"{accession}.txt"
    names = zf.namelist()

    # Most entries are nested in folders in the ZIP
    cand = [n for n in names if n.endswith("/" + target)]
    if not cand:
        cand = [n for n in names if n.endswith(target)]
    if not cand:
        raise FileNotFoundError(f"{target} was not found inside the MassBank-data ZIP.")

    with zf.open(cand[0], "r") as fh:
        return fh.read().decode("utf-8", errors="replace")


# =============================================================================
# Parsing utilities (PK$PEAK, PK$ANNOTATION, metadata)
# =============================================================================

FLOAT = r"[-+]?(?:\d+(?:[.,]\d+)?(?:[eE][-+]?\d+)?)"
sect_re = re.compile(r"^(CH|AC|MS|PK|COMMENT)\$", re.IGNORECASE)
num_re  = re.compile(FLOAT)

def parse_pk_peak(txt: str) -> List[Tuple[float, float, Optional[float]]]:
    """
    Parse the PK$PEAK section.

    Returns:
        A list of tuples: (mz, abs_intensity, rel_intensity)
        If rel_intensity is not present, it is computed as 100*abs/max(abs).
    """
    out: List[Tuple[float, float, Optional[float]]] = []
    in_pk = False

    for line in txt.splitlines():
        s = line.strip()
        if not in_pk:
            if s.startswith("PK$PEAK"):
                in_pk = True
            continue

        # Stop when we reach a new section or empty line
        if not s or sect_re.match(s):
            break

        nums = [n.replace(",", ".") for n in num_re.findall(s)]
        if len(nums) >= 3:
            mz, abs_i, rel_i = float(nums[0]), float(nums[1]), float(nums[2])
            out.append((mz, abs_i, rel_i))
        elif len(nums) >= 2:
            mz, abs_i = float(nums[0]), float(nums[1])
            out.append((mz, abs_i, None))

    # If all rel_intensity values are missing, compute them
    if out and all(t[2] is None for t in out):
        mx = max(t[1] for t in out) or 1.0
        out = [(mz, ai, 100.0 * ai / mx) for (mz, ai, _) in out]

    return out

def parse_pk_annotation(txt: str) -> Dict[float, str]:
    """
    Parse the PK$ANNOTATION section and extract formulas associated with m/z.

    Returns:
        A mapping {annotated_mz: "formula", ...}
    """
    ann: Dict[float, str] = {}
    in_ann = False

    for line in txt.splitlines():
        s = line.strip()
        if not in_ann:
            if s.startswith("PK$ANNOTATION"):
                in_ann = True
            continue

        if not s or sect_re.match(s):
            break

        m = num_re.search(s.replace(",", "."))
        if not m:
            continue

        mz = float(m.group(0))

        # Try a few common patterns to extract formula text
        mf = re.search(
            r"(?:tentative\s+formula|formula)\s*[:=]\s*([A-Za-z0-9+\-()\.]+)",
            s,
            flags=re.IGNORECASE
        )
        if mf:
            ann[mz] = mf.group(1)
        else:
            # Fallback: scan tokens from the end and pick something formula-like
            tokens = re.split(r"[\t ]+", s)
            for t in reversed(tokens):
                if re.fullmatch(r"[A-Za-z][A-Za-z0-9()+\-\.]*", t) and any(c.isdigit() for c in t):
                    ann[mz] = t
                    break

    return ann

def ppm(a: float, b: float) -> float:
    """
    Compute the absolute mass error in parts-per-million (ppm).
    """
    return abs(a - b) / a * 1e6 if a else float("inf")

def attach_formula(df_out: pd.DataFrame, ann: Dict[float, str], ppm_tol: float) -> pd.DataFrame:
    """
    Attach formulas to peaks by matching each peak m/z to the closest annotated m/z.

    Args:
        df_out: DataFrame with at least a 'mz' column.
        ann: Mapping {annotated_mz: formula}.
        ppm_tol: Maximum allowed ppm error to accept a match.

    Returns:
        df_out with an extra 'formula' column (empty string if no match).
    """
    if not ann:
        df_out["formula"] = ""
        return df_out

    items = sorted(ann.items())
    formulas: List[str] = []

    for mz in df_out["mz"].to_numpy():
        best = ("", 1e9)  # (formula, ppm_error)
        for mz_pa, f in items:
            d = ppm(float(mz), float(mz_pa))
            if d < best[1]:
                best = (f, d)
            # Small early-stop heuristic
            if mz_pa > mz and best[1] <= ppm_tol and d > best[1]:
                break
        formulas.append(best[0] if best[1] <= ppm_tol else "")

    df_out["formula"] = formulas
    return df_out

def parse_metadata(txt: str) -> Dict[str, str]:
    """
    Extract a subset of useful metadata fields from the MassBank record text.

    Returns:
        A dict containing keys like:
        ACCESSION, RECORD_TITLE, CH$NAME, instrument, ion mode, collision energy, etc.
    """
    meta: Dict[str, str] = {}
    patterns = {
        "ACCESSION":      r"^ACCESSION:\s*(\S+)",
        "RECORD_TITLE":   r"^RECORD_TITLE:\s*(.+)",
        "CH$NAME":        r"^CH\$NAME:\s*(.+)",
        "CH$FORMULA":     r"^CH\$FORMULA:\s*(\S+)",
        "CH$EXACT_MASS":  r"^CH\$EXACT_MASS:\s*([0-9\.\-Ee]+)",
        "AC$INSTRUMENT":  r"^AC\$INSTRUMENT:\s*(.+)",
        "AC$INSTRUMENT_TYPE": r"^AC\$INSTRUMENT_TYPE:\s*(.+)",
        "AC$MASS_SPECTROMETRY: MS_TYPE": r"^AC\$MASS_SPECTROMETRY:\s*MS_TYPE\s*(.+)",
        "AC$MASS_SPECTROMETRY: ION_MODE": r"^AC\$MASS_SPECTROMETRY:\s*ION_MODE\s*(.+)",
        "AC$MASS_SPECTROMETRY: COLLISION_ENERGY": r"^AC\$MASS_SPECTROMETRY:\s*COLLISION_ENERGY\s*(.+)",
        "AC$MASS_SPECTROMETRY: RESOLUTION": r"^AC\$MASS_SPECTROMETRY:\s*RESOLUTION\s*(.+)",
    }
    for key, pat in patterns.items():
        m = re.search(pat, txt, flags=re.MULTILINE)
        if m:
            meta[key] = m.group(1).strip()
    return meta


# =============================================================================
# Naming / README helpers
# =============================================================================

def molecule_name_from_meta(meta: Dict[str, str]) -> str:
    """
    Build a safe molecule name for filenames/README.

    Priority:
        - CH$NAME if available
        - else: first chunk of RECORD_TITLE before ';'
    """
    mol = (meta.get("CH$NAME") or "").strip()
    if not mol:
        rt = meta.get("RECORD_TITLE", "") or ""
        mol = rt.split(";")[0].strip() if ";" in rt else rt.strip()

    mol = re.sub(r"[^\w\.\-\+]+", "_", mol).strip("_")
    mol = re.sub(r"_+", "_", mol)
    return mol or "UnknownMolecule"

def sanitize_filename(s: str) -> str:
    """
    Sanitize a string to be safe for filenames across OSes.
    """
    s = re.sub(r"[^\w\.\-\+]+", "_", s.strip())
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:160]

def build_descriptive_basename(meta: Dict[str, str]) -> str:
    """
    Build a descriptive base filename using metadata fields.
    Fallback used if RECORD_TITLE parsing fails.
    """
    name   = meta.get("CH$NAME") or ""
    ion    = meta.get("AC$MASS_SPECTROMETRY: ION_MODE") or ""
    ms_ty  = meta.get("AC$MASS_SPECTROMETRY: MS_TYPE") or ""
    ce     = meta.get("AC$MASS_SPECTROMETRY: COLLISION_ENERGY") or ""
    instr  = meta.get("AC$INSTRUMENT") or meta.get("AC$INSTRUMENT_TYPE") or ""
    acc    = meta.get("ACCESSION") or "unknown"

    parts: List[str] = []
    if name:  parts.append(name)
    if instr: parts.append(instr)
    if ms_ty: parts.append(ms_ty)
    if ion:   parts.append(ion)
    if ce:    parts.append(ce.replace(" ", ""))
    parts.append(acc)

    base = "__".join(sanitize_filename(p) for p in parts if p)
    return base or sanitize_filename(acc)

def name_from_record_title(meta: Dict[str, str]) -> str:
    """
    Build a compact filename based on RECORD_TITLE, e.g.:
        Triadimefon_LC_ESI_QFT_MS2_CE_30

    Example RECORD_TITLE:
        Triadimefon; LC-ESI-QFT; MS2; CE: 30%; R=17500; [M+H]+
    """
    title = meta.get("RECORD_TITLE", "") or ""
    m = re.search(
        r'^\s*([^;]+)\s*;\s*([^;]+)\s*;\s*(MS\d+)\s*;\s*CE:\s*([\d.]+)',
        title,
        flags=re.IGNORECASE
    )
    if m:
        mol, instr, ms, ce = m.groups()
        instr = instr.replace("-", "_").replace(" ", "_")
        mol   = mol.strip().replace(" ", "_")

        base  = f"{mol}_{instr}_{ms}_CE_{ce}"
        base  = re.sub(r'[^A-Za-z0-9_]+', '_', base)
        base  = re.sub(r'_+', '_', base).strip('_')
        return base

    return build_descriptive_basename(meta)

def render_readme_section(
    meta: Dict[str, str],
    n_peaks: int,
    n_with_formula: int,
    ppm_tol: float,
    dat_filename: str
) -> str:
    """
    Create a Markdown snippet describing one exported record.

    This is appended into a per-molecule README file.
    """
    now = _dt.datetime.now().isoformat(timespec="seconds")

    title = (meta.get("RECORD_TITLE", "") or "").strip()
    acc   = (meta.get("ACCESSION", "") or "").strip()

    head  = f"## {title}" + (f" ({acc})" if acc else "")

    lines: List[str] = []
    lines += [
        head, "",
        f"- Generated: {now}",
        f"- .dat file: `{dat_filename}`",
    ]

    for k in [
        "CH$NAME", "CH$FORMULA", "CH$EXACT_MASS",
        "AC$INSTRUMENT", "AC$INSTRUMENT_TYPE",
        "AC$MASS_SPECTROMETRY: MS_TYPE",
        "AC$MASS_SPECTROMETRY: ION_MODE",
        "AC$MASS_SPECTROMETRY: COLLISION_ENERGY",
        "AC$MASS_SPECTROMETRY: RESOLUTION",
    ]:
        if k in meta and meta[k]:
            lines.append(f"- {k}: {meta[k]}")

    lines += [
        f"- Peaks: {n_peaks} (intensity = relative %)",
        f"- With formula matched (±{ppm_tol} ppm): {n_with_formula}",
        "",
        f"Columns exported in `{dat_filename}`: `intensity` (rel%), `mz`, `formula`.",
        "",
        "---", ""
    ]
    return "\n".join(lines)

"""
╔═══════════════════════════════════════════════╗
║        +                                      ║
║        +                Mass                  ║
║        +                 |                    ║
║        +    Customizable |                    ║
║        +          |      |                    ║
║        +          |      |   Spectʀa          ║
║        +          |      |      |             ║
║        +          |      |      |             ║
║        +++++++++++++++++++++++++++++++++      ║
╚═══════════════════════════════════════════════╝
"""
def _safe_filename(s: str, maxlen: int = 140) -> str:
    """Make a Windows-safe filename from RECORD_TITLE."""
    s = s.strip()
    # Replace forbidden Windows filename chars: \ / : * ? " < > |
    s = re.sub(r'[\\/:*?"<>|]+', "_", s)
    s = re.sub(r"\s+", " ", s)  # collapse whitespace
    s = s.replace("\n", " ").strip()
    return s[:maxlen].rstrip(" .")

def plot_with_filter(
    df_plot,
    title: str,
    min_mz: float = 0.0,
    filter_threshold: float = 0.0,
    highlight_color: str = "orange",
):
    """
    Interactive MS stick spectrum (requires %matplotlib widget + ipympl).

    Hover behavior:
    - Vertical guide line follows mouse
    - Nearest peak is highlighted (different color)
    - Tooltip shows: Mass=X, Rel. Intensity=Y
    """
    df = df_plot.copy()
    df["mz"] = pd.to_numeric(df["mz"], errors="coerce")
    df["intensity"] = pd.to_numeric(df["intensity"], errors="coerce")
    df = df.dropna(subset=["mz", "intensity"])
    df = df[df["mz"] >= float(min_mz)]

    if df.empty:
        print("No peaks to plot.")
        return None

    # Threshold as % of max intensity
    max_i = float(df["intensity"].max())
    thr = (float(filter_threshold) * max_i) / 100.0
    df = df[df["intensity"] >= thr].copy()
    if df.empty:
        print("No peaks above threshold.")
        return None

    df = df.sort_values("mz")
    x = df["mz"].to_numpy(dtype=float)
    y = df["intensity"].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(9, 4))

    # Base spectrum (all peaks)
    ax.vlines(x, 0, y, linewidth=1.3)

    # Highlight line (single peak, moved on hover)
    hi = ax.vlines([x[0]], 0, [y[0]], linewidth=2.6, colors=highlight_color)

    # Vertical guide line
    guide = ax.axvline(x[0], linewidth=1.0, alpha=0.25)

    # Annotation (tooltip-like)
    ann = ax.annotate(
        "",
        xy=(x[0], y[0]),
        xytext=(12, 12),
        textcoords="offset points",
        bbox=dict(boxstyle="round,pad=0.25", alpha=0.85),
        arrowprops=dict(arrowstyle="->", alpha=0.5),
    )
    ann.set_visible(False)

    ax.set_title(title)
    ax.set_xlabel("m/z")
    ax.set_ylabel("Relative intensity (%)")
    ax.set_ylim(0, max(100, y.max() * 1.15))

    # Make single-peak spectra still look like spectra (margin in x)
    if np.isclose(x.min(), x.max()):
        margin = max(5.0, x.min() * 0.02)
        ax.set_xlim(x.min() - margin, x.max() + margin)
    else:
        margin = (x.max() - x.min()) * 0.03
        ax.set_xlim(x.min() - margin, x.max() + margin)

    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.yaxis.grid(True, alpha=0.2, linestyle="--")
    fig.tight_layout()

    def on_move(event):
        if event.inaxes != ax or event.xdata is None:
            ann.set_visible(False)
            fig.canvas.draw_idle()
            return

        # Guide line follows cursor
        guide.set_xdata([event.xdata, event.xdata])

        # Nearest peak
        idx = int(np.argmin(np.abs(x - event.xdata)))

        # Move highlight line to that peak
        hi.set_segments([[(x[idx], 0.0), (x[idx], y[idx])]])

        # Update annotation
        ann.xy = (x[idx], y[idx])
        ann.set_text(f"Mass = {x[idx]:.4f}\nRel. Intensity = {y[idx]:.2f}")
        ann.set_visible(True)

        fig.canvas.draw_idle()

    cid = fig.canvas.mpl_connect("motion_notify_event", on_move)

    # Return objects so you can save later without replotting
    return {"fig": fig, "ax": ax, "cid": cid, "x": x, "y": y}
                
def plot_mass_spectrum_plotly(df_plot, title="Mass spectrum", min_mz=0.0, filter_threshold=0.0):
    df = df_plot.copy()
    df = df[df["mz"] >= float(min_mz)]
    if df.empty:
        return go.Figure().update_layout(title=title)

    max_i = df["intensity"].max()
    thr = (float(filter_threshold) * float(max_i)) / 100.0
    df = df[df["intensity"] >= thr].copy()
    df = df.sort_values("mz")

    # Hover text (añade fórmula si existe)
    if "formula" in df.columns:
        hover = [
            f"m/z: {mz:.4f}<br>Intensity: {inten:.2f}<br>Formula: {form}"
            for mz, inten, form in zip(df["mz"], df["intensity"], df["formula"].fillna(""))
        ]
    else:
        hover = [f"m/z: {mz:.4f}<br>Intensity: {inten:.2f}" for mz, inten in zip(df["mz"], df["intensity"])]

    fig = go.Figure()

    # Picos como líneas verticales (tipo espectro real)
    fig.add_trace(go.Scatter(
        x=df["mz"],
        y=df["intensity"],
        mode="lines+markers",
        line=dict(width=1),
        marker=dict(size=6),
        text=hover,
        hoverinfo="text"
    ))

    fig.update_layout(
        title=title,
        xaxis_title="m/z",
        yaxis_title="Relative intensity (%)",
        hovermode="closest"
    )
    return fig

# =============================================================================
# Jupyter UI (ipywidgets)
# =============================================================================

txt_name = W.Text(
    value=DEFAULT_MOLECULE,
    description="Molecule:",
    layout=W.Layout(width="60%")
)

btn_search = W.Button(
    description="Search MS2",
    button_style="primary"
)

ta_access = W.Textarea(
    value="",
    description="Accessions\n(optional):",
    layout=W.Layout(width="60%", height="100px")
)

ppm_box = W.BoundedFloatText(
    value=PPM_TOL_DEFAULT,
    min=0.1,
    max=50.0,
    step=0.1,
    description="ppm tol:"
)

zip_path = W.Text(
    value=LOCAL_ZIP_PATH,
    description="Local ZIP:",
    placeholder="Leave empty to download GitHub main.zip",
    layout=W.Layout(width="60%")
)

btn_load = W.Button(
    description="Load selected",
    button_style="info",
    disabled=True
)

btn_export = W.Button(
    description="Export .dat + README",
    button_style="success",
    disabled=True
)

out = W.Output()
plot_out = W.Output()
table_out = W.Output()

dd_select = W.SelectMultiple(
    options=[],
    description="MS2:",
    rows=12,
    layout=W.Layout(width="60%")
)



def on_search(_):
    """
    UI callback: search MS2 records by molecule name and populate the selection list.
    """
    out.clear_output()
    dd_select.options = []

    with out:
        name = txt_name.value.strip()

        if not name:
            print("Please enter a molecule name or paste accessions below.")
            btn_load.disabled = False
            return

        print(f"Searching MS2 records for '{name}' ...")

        try:
            accs = api_search_ms2_by_name(name)
        except Exception as e:
            print("API call failed:", e)
            accs = []

        if not accs:
            print("No records found via API (or network error).")
            print("You can paste accessions manually (one per line) in the box above.")
            btn_load.disabled = True
        else:
            dd_select.options = [(f"{a['accession']} — {a['title']}", a["accession"]) for a in accs]
            print(f"Found {len(dd_select.options)} records. Select entries and click 'Load selected'.")
            btn_load.disabled = False


def on_load(_):
    """
    UI callback: load selected (or manually pasted) accessions, parse spectra, and show a preview.
    """
    out.clear_output()

    with out:
        sel = list(dd_select.value)
        manual = [s.strip() for s in ta_access.value.splitlines() if s.strip()]
        manual = [m.replace("MSBNK-", "") for m in manual]
        accessions = sel or manual

        if ta_access.value.strip():
            btn_load.disabled = False

        if not accessions:
            print("Please select entries or paste accessions first.")
            return

        try:
            zf = open_massbank_zip(zip_path.value.strip() or None)
        except Exception as e:
            print("Could not open/download the MassBank-data ZIP:", e)
            return

        rows_all = []
        metas = []
        ppm_tol = float(ppm_box.value)

        for i, acc in enumerate(accessions, start=1):
            try:
                txt = read_accession_txt(zf, acc)
            except Exception as e:
                print(f"[{acc}] not found in ZIP:", e)
                continue

            pk = parse_pk_peak(txt)
            if not pk:
                print(f"[{acc}] missing PK$PEAK section.")
                continue

            ann = parse_pk_annotation(txt)
            meta = parse_metadata(txt)

            df = pd.DataFrame(pk, columns=["mz", "abs_i", "rel_i"])
            df_out = df.assign(intensity=df["rel_i"].astype(float))[["intensity", "mz"]]
            df_out = attach_formula(df_out, ann, ppm_tol)

            # Sort by intensity (desc) and then by m/z (asc)
            df_out = df_out.sort_values(
                ["intensity", "mz"],
                ascending=[False, True],
                kind="mergesort"
            ).reset_index(drop=True)

            df_out["accession"] = acc
            rows_all.append(df_out)

            metas.append({
                "accession": acc,
                "n_peaks": int(df_out.shape[0]),
                "with_formula": int((df_out["formula"].astype(str).str.len() > 0).sum()),
                "RECORD_TITLE": meta.get("RECORD_TITLE", ""),
                "ION_MODE": meta.get("AC$MASS_SPECTROMETRY: ION_MODE", ""),
                "CE": meta.get("AC$MASS_SPECTROMETRY: COLLISION_ENERGY", ""),
            })

        if not rows_all:
            print("No data could be loaded for the selected/pasted accessions.")
            return

        global _DF_ALL, _METAS, _ZIP_USED
        _DF_ALL = pd.concat(rows_all, ignore_index=True)
        _METAS = metas
        _ZIP_USED = (zip_path.value.strip() or "GitHub main.zip")

        print("Summary:")
        display(pd.DataFrame(metas))

        print("\nPreview (first rows):")
        display(_DF_ALL.head(40))

        btn_export.disabled = False
            
        plot_out.clear_output(wait=True)
        with plot_out:
            df_plot = _DF_ALL[_DF_ALL["accession"] == accessions[0]].copy()
            record_title = metas[0].get("RECORD_TITLE", accessions[0])

            _PLOT_STATE = plot_with_filter(
                df_plot,
                filter_threshold=1,
                min_mz=0,
                title=metas[0]["RECORD_TITLE"]
            )
             # En ipympl lo más fiable es mostrar el canvas:
            if _PLOT_STATE is not None:
                display(_PLOT_STATE["fig"].canvas)
                #display(fig.canvas)

def on_export(_):
    """
    UI callback: export .dat files (one per accession) and per-molecule README logs.
    """
    out.clear_output()

    with out:
        try:
            DF = _DF_ALL.copy()
            metas = _METAS
        except NameError:
            print("No data loaded yet. Click 'Load selected' first.")
            return

        os.makedirs(OUT_DIR, exist_ok=True)
        ppm_tol = float(ppm_box.value)

        exported = []
        readmes = []

        # Re-open ZIP to read full metadata again for the export step
        try:
            zf = open_massbank_zip(zip_path.value.strip() or None)
        except Exception as e:
            print("Could not open ZIP for README export:", e)
            return

        for acc, g in DF.groupby("accession", sort=False):
            txt = read_accession_txt(zf, acc)
            meta = parse_metadata(txt)

            base = name_from_record_title(meta)
            dat_path = os.path.join(OUT_DIR, f"{base}.dat")

            # Ensure sorted output
            g_sorted = g.sort_values(
                ["intensity", "mz"],
                ascending=[False, True],
                kind="mergesort"
            ).reset_index(drop=True)

            g_sorted[["intensity", "mz", "formula"]].to_csv(dat_path, sep="\t", index=False)
            exported.append(dat_path)

            n_peaks = int(g_sorted.shape[0])
            n_with_formula = int((g_sorted["formula"].astype(str).str.len() > 0).sum())

            dat_filename = os.path.basename(dat_path)
            readme_section = render_readme_section(meta, n_peaks, n_with_formula, ppm_tol, dat_filename)

            mol_name = molecule_name_from_meta(meta)
            mol_readme_path = os.path.join(OUT_DIR, f"{mol_name}_README.md")

            # Create header once
            if not os.path.exists(mol_readme_path):
                with open(mol_readme_path, "w", encoding="utf-8") as fh:
                    fh.write(f"# {mol_name} — Spectra log\n\n")

            # Avoid duplicate entries by ACCESSION
            acc_id = meta.get("ACCESSION", "")
            existing = ""
            try:
                with open(mol_readme_path, "r", encoding="utf-8") as fh:
                    existing = fh.read()
            except FileNotFoundError:
                pass

            if not (acc_id and acc_id in existing):
                with open(mol_readme_path, "a", encoding="utf-8") as fh:
                    fh.write(readme_section)

            readmes.append(mol_readme_path)

        print("Exported files:")
        for p in exported:
            print(" -", p)

        print("README logs:")
        for p in readmes:
            print(" -", p)


# Wire callbacks
btn_search.on_click(on_search)
btn_load.on_click(on_load)
btn_export.on_click(on_export)

# Render UI
display(W.HBox([txt_name, btn_search]))
display(dd_select)
display(ta_access)
display(W.HBox([ppm_box]))
display(W.HBox([zip_path]))
display(W.HBox([btn_load, btn_export]))
display(W.HBox([out, plot_out]))


SelectMultiple(description='MS2:', layout=Layout(width='60%'), options=(), rows=12, value=())

Textarea(value='', description='Accessions\n(optional):', layout=Layout(height='100px', width='60%'))